### Advanced Read-Only and Computed Properties — Tutorial Problems

In this notebook we are going to continue working with computed properties, but we are going to push the idea quite a bit further.

We will still start from the same basic idea:

- some values belong naturally to an object,
- those values do not necessarily need their own backing variables,
- and a property can calculate them from other state.

But now we will add some harder issues:

- validation,
- dependent computed properties,
- caching,
- invalidating only the cache that actually became stale,
- caching values where `None` is a valid result,
- exposing internal collections safely,
- lazy parsing,
- explicit refresh operations,
- and version-based cache management.

The important thing is not just to write the final class.

For every problem we will first build a simpler version, inspect its behavior, identify a weakness, and then improve it.

That is often the easiest way to understand why a more advanced design is useful in the first place.

### Problem 1 — A Computed Bounding Box

Suppose we have a collection of 2D points.

We want a class that stores those points, but also gives us some useful geometric information:

- minimum `x`
- maximum `x`
- minimum `y`
- maximum `y`
- width
- height
- area of the bounding box

These values are all derived from the points.

So our first instinct should be that we probably do **not** want to store each one separately.

Let's start with a very small version.

We will store the points as an immutable tuple, and calculate the bounding box values every time they are requested.

In [1]:
class PointCloud:
    def __init__(self, points):
        self.points = tuple(points)

    @property
    def min_x(self):
        print("calculating min_x...")
        return min(x for x, y in self.points)

    @property
    def max_x(self):
        print("calculating max_x...")
        return max(x for x, y in self.points)

    @property
    def min_y(self):
        print("calculating min_y...")
        return min(y for x, y in self.points)

    @property
    def max_y(self):
        print("calculating max_y...")
        return max(y for x, y in self.points)

Let's create a point cloud and inspect the four values.

In [2]:
cloud = PointCloud([
    (2, 5),
    (10, 3),
    (-1, 8),
    (4, -2),
])

cloud.min_x, cloud.max_x, cloud.min_y, cloud.max_y

calculating min_x...
calculating max_x...
calculating min_y...
calculating max_y...


(-1, 10, -2, 8)

Now we can add a few more computed properties.

Notice that `width` and `height` depend on other computed properties.

That is perfectly legal.

In [3]:
class PointCloud:
    def __init__(self, points):
        self.points = tuple(points)

    @property
    def min_x(self):
        print("calculating min_x...")
        return min(x for x, y in self.points)

    @property
    def max_x(self):
        print("calculating max_x...")
        return max(x for x, y in self.points)

    @property
    def min_y(self):
        print("calculating min_y...")
        return min(y for x, y in self.points)

    @property
    def max_y(self):
        print("calculating max_y...")
        return max(y for x, y in self.points)

    @property
    def width(self):
        return self.max_x - self.min_x

    @property
    def height(self):
        return self.max_y - self.min_y

    @property
    def bounding_area(self):
        return self.width * self.height

In [4]:
cloud = PointCloud([
    (2, 5),
    (10, 3),
    (-1, 8),
    (4, -2),
])

cloud.bounding_area

calculating max_x...
calculating min_x...
calculating max_y...
calculating min_y...


110

This works, but there is an important detail hiding in the output.

To calculate `bounding_area` we calculate:

- `width`
- which calculates `max_x`
- and then `min_x`

and then:

- `height`
- which calculates `max_y`
- and then `min_y`

That means a single access to `bounding_area` scans the points four separate times.

For a tiny collection this does not matter.

For a point cloud containing millions of points it might matter a lot.

We can improve this by calculating the entire bounding box once and caching it.

Instead of caching four separate values, we are going to cache one tuple:

```python
(min_x, max_x, min_y, max_y)
```

This is useful because all four values come from the same scan of the data.

In [5]:
class PointCloud:
    def __init__(self, points):
        self._points = ()
        self._bounds = None
        self.points = points

    @property
    def points(self):
        return self._points

    @points.setter
    def points(self, value):
        value = tuple(value)

        if not value:
            raise ValueError("a point cloud cannot be empty")

        for point in value:
            if not (
                isinstance(point, tuple)
                and len(point) == 2
            ):
                raise TypeError("each point must be a 2-tuple")

        self._points = value
        self._bounds = None

    def _calculate_bounds(self):
        print("calculating all bounds...")

        xs = [x for x, y in self.points]
        ys = [y for x, y in self.points]

        self._bounds = (
            min(xs),
            max(xs),
            min(ys),
            max(ys),
        )

    @property
    def bounds(self):
        if self._bounds is None:
            self._calculate_bounds()

        return self._bounds

The `bounds` property is read-only.

The user can replace the points, but cannot assign directly to `bounds`.

Let's check that the bounds are only calculated once.

In [6]:
cloud = PointCloud([
    (2, 5),
    (10, 3),
    (-1, 8),
    (4, -2),
])

cloud.bounds

calculating all bounds...


(-1, 10, -2, 8)

In [7]:
cloud.bounds

(-1, 10, -2, 8)

Now let's build the other computed properties on top of the cached bounds.

In [8]:
class PointCloud:
    def __init__(self, points):
        self._points = ()
        self._bounds = None
        self.points = points

    @property
    def points(self):
        return self._points

    @points.setter
    def points(self, value):
        value = tuple(value)

        if not value:
            raise ValueError("a point cloud cannot be empty")

        for point in value:
            if not (
                isinstance(point, tuple)
                and len(point) == 2
            ):
                raise TypeError("each point must be a 2-tuple")

        if value != self._points:
            self._points = value
            self._bounds = None

    def _calculate_bounds(self):
        print("calculating all bounds...")

        xs = [x for x, y in self.points]
        ys = [y for x, y in self.points]

        self._bounds = (
            min(xs),
            max(xs),
            min(ys),
            max(ys),
        )

    @property
    def bounds(self):
        if self._bounds is None:
            self._calculate_bounds()
        return self._bounds

    @property
    def min_x(self):
        return self.bounds[0]

    @property
    def max_x(self):
        return self.bounds[1]

    @property
    def min_y(self):
        return self.bounds[2]

    @property
    def max_y(self):
        return self.bounds[3]

    @property
    def width(self):
        return self.max_x - self.min_x

    @property
    def height(self):
        return self.max_y - self.min_y

    @property
    def bounding_area(self):
        return self.width * self.height

In [9]:
cloud = PointCloud([
    (0, 0),
    (10, 5),
    (3, 12),
])

print(cloud.width)
print(cloud.height)
print(cloud.bounding_area)

calculating all bounds...
10
12
120


Only the first access needs to calculate the bounds.

After that, `width`, `height`, and `bounding_area` can all use the cached result.

But we also need to make sure the cache becomes invalid if the points change.

In [10]:
cloud.points = [
    (-5, -5),
    (5, 5),
]

cloud.bounding_area

calculating all bounds...


100

The assignment to `points` invalidated `_bounds`.

The next computed-property access then rebuilt the cache from the new data.

This is one of the most common patterns for computed properties:

1. store the independent state,
2. derive other values,
3. cache expensive derived values,
4. invalidate those cached values when the independent state changes.

### Problem 2 — A Shopping Cart with Selective Cache Invalidation

Let's make the cache problem harder.

Suppose a shopping cart has:

- line items,
- a discount rate,
- a shipping fee.

We want these read-only computed properties:

- `subtotal`
- `discount_amount`
- `discounted_subtotal`
- `total`

The interesting part is that changing one input does not necessarily make **every** cached value stale.

For example, changing the shipping fee does not change the subtotal.

So it would be wasteful to throw away the subtotal cache every time shipping changes.

Let's first write a simple version without any caching.

In [11]:
class ShoppingCart:
    def __init__(self, items, discount_rate=0.0, shipping_fee=0.0):
        self.items = tuple(items)
        self.discount_rate = discount_rate
        self.shipping_fee = shipping_fee

    @property
    def subtotal(self):
        print("calculating subtotal...")
        return sum(price * quantity for price, quantity in self.items)

    @property
    def discount_amount(self):
        print("calculating discount...")
        return self.subtotal * self.discount_rate

    @property
    def discounted_subtotal(self):
        return self.subtotal - self.discount_amount

    @property
    def total(self):
        return self.discounted_subtotal + self.shipping_fee

In [12]:
cart = ShoppingCart(
    items=[
        (10.0, 2),
        (5.0, 3),
    ],
    discount_rate=0.10,
    shipping_fee=4.0,
)

cart.total

calculating subtotal...
calculating discount...
calculating subtotal...


35.5

Again, the result is correct.

But notice how many times `subtotal` can be recalculated while building one final answer.

We will now cache the derived values.

This time we are going to keep separate caches because different inputs affect different results.

The dependency relationships are:

```text
items
  └── subtotal
        ├── discount_amount
        │      └── discounted_subtotal
        │              └── total
        └── discounted_subtotal
               └── total

discount_rate
  └── discount_amount
         └── discounted_subtotal
                └── total

shipping_fee
  └── total
```

So we want invalidation to follow these dependencies.

In [13]:
_NOT_CACHED = object()

class ShoppingCart:
    def __init__(self, items, discount_rate=0.0, shipping_fee=0.0):
        self._items = ()
        self._discount_rate = None
        self._shipping_fee = None

        self._subtotal = _NOT_CACHED
        self._discount_amount = _NOT_CACHED
        self._discounted_subtotal = _NOT_CACHED
        self._total = _NOT_CACHED

        self.items = items
        self.discount_rate = discount_rate
        self.shipping_fee = shipping_fee

    def _invalidate_from_items(self):
        self._subtotal = _NOT_CACHED
        self._discount_amount = _NOT_CACHED
        self._discounted_subtotal = _NOT_CACHED
        self._total = _NOT_CACHED

    def _invalidate_from_discount(self):
        self._discount_amount = _NOT_CACHED
        self._discounted_subtotal = _NOT_CACHED
        self._total = _NOT_CACHED

    def _invalidate_from_shipping(self):
        self._total = _NOT_CACHED

At this point we only created the storage and invalidation helpers.

Now we can add the writable input properties one at a time.

In [14]:
def _validate_money(value, name):
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise TypeError(f"{name} must be numeric")
    if value < 0:
        raise ValueError(f"{name} cannot be negative")


class ShoppingCart(ShoppingCart):
    @property
    def items(self):
        return self._items

    @items.setter
    def items(self, value):
        value = tuple(value)

        for price, quantity in value:
            _validate_money(price, "price")

            if (
                isinstance(quantity, bool)
                or not isinstance(quantity, int)
                or quantity <= 0
            ):
                raise ValueError("quantity must be a positive integer")

        if value != self._items:
            self._items = value
            self._invalidate_from_items()

    @property
    def discount_rate(self):
        return self._discount_rate

    @discount_rate.setter
    def discount_rate(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("discount_rate must be numeric")

        if not 0 <= value <= 1:
            raise ValueError("discount_rate must be between 0 and 1")

        if value != self._discount_rate:
            self._discount_rate = value
            self._invalidate_from_discount()

    @property
    def shipping_fee(self):
        return self._shipping_fee

    @shipping_fee.setter
    def shipping_fee(self, value):
        _validate_money(value, "shipping_fee")

        if value != self._shipping_fee:
            self._shipping_fee = value
            self._invalidate_from_shipping()

Notice that changing `shipping_fee` only invalidates `_total`.

That is safe because shipping has no effect on the other calculations.

Now we can add the cached computed properties.

In [15]:
class ShoppingCart(ShoppingCart):
    @property
    def subtotal(self):
        if self._subtotal is _NOT_CACHED:
            print("calculating subtotal...")
            self._subtotal = sum(
                price * quantity
                for price, quantity in self.items
            )
        return self._subtotal

    @property
    def discount_amount(self):
        if self._discount_amount is _NOT_CACHED:
            print("calculating discount amount...")
            self._discount_amount = self.subtotal * self.discount_rate
        return self._discount_amount

    @property
    def discounted_subtotal(self):
        if self._discounted_subtotal is _NOT_CACHED:
            print("calculating discounted subtotal...")
            self._discounted_subtotal = (
                self.subtotal - self.discount_amount
            )
        return self._discounted_subtotal

    @property
    def total(self):
        if self._total is _NOT_CACHED:
            print("calculating total...")
            self._total = self.discounted_subtotal + self.shipping_fee
        return self._total

In [16]:
cart = ShoppingCart(
    items=[
        (10.0, 2),
        (5.0, 3),
    ],
    discount_rate=0.10,
    shipping_fee=4.0,
)

cart.total

calculating total...
calculating discounted subtotal...
calculating subtotal...
calculating discount amount...


35.5

In [17]:
cart.total

35.5

The second access is fully cached.

Now let's change only the shipping fee.

In [18]:
cart.shipping_fee = 10.0
cart.total

calculating total...


41.5

Only `total` needed to be recalculated.

The subtotal, discount amount, and discounted subtotal were still valid, so we kept them.

Now change the discount rate.

In [19]:
cart.discount_rate = 0.20
cart.total

calculating total...
calculating discounted subtotal...
calculating discount amount...


38.0

This time we kept the subtotal, but the discount-related values and the final total had to be rebuilt.

This is called **selective cache invalidation**.

It is more complicated than simply clearing every cache, but it can be useful when some calculations are expensive.

### Problem 3 — When `None` Is a Valid Cached Result

So far we have often used `None` to mean:

> this value has not been calculated yet

That works only if `None` can never be a legitimate result.

Let's look at a case where `None` is meaningful.

Suppose a temperature log stores readings, and we want a read-only property called `first_freezing_reading`.

It should return:

- the first reading at or below `0`,
- or `None` if no such reading exists.

Here is a first attempt.

In [20]:
class TemperatureLog:
    def __init__(self, readings):
        self.readings = tuple(readings)
        self._first_freezing = None

    @property
    def first_freezing_reading(self):
        if self._first_freezing is None:
            print("searching for freezing reading...")

            self._first_freezing = next(
                (value for value in self.readings if value <= 0),
                None,
            )

        return self._first_freezing

In [21]:
log = TemperatureLog([12, 8, 5, 3])

log.first_freezing_reading

searching for freezing reading...


In [22]:
log.first_freezing_reading

searching for freezing reading...


We have a bug.

The correct result is `None`, but `None` is also our marker for “not calculated yet”.

So the property repeats the search every time.

The usual solution is to create a unique sentinel object.

That object means “not calculated yet”, while `None` can remain a real cached value.

In [23]:
_NOT_CALCULATED = object()

class TemperatureLog:
    def __init__(self, readings):
        self._readings = ()
        self._first_freezing = _NOT_CALCULATED
        self.readings = readings

    @property
    def readings(self):
        return self._readings

    @readings.setter
    def readings(self, value):
        value = tuple(value)

        if value != self._readings:
            self._readings = value
            self._first_freezing = _NOT_CALCULATED

    @property
    def first_freezing_reading(self):
        if self._first_freezing is _NOT_CALCULATED:
            print("searching for freezing reading...")

            self._first_freezing = next(
                (value for value in self.readings if value <= 0),
                None,
            )

        return self._first_freezing

In [24]:
log = TemperatureLog([12, 8, 5, 3])

log.first_freezing_reading

searching for freezing reading...


In [25]:
log.first_freezing_reading

Now the second access uses the cached `None`.

There is no repeated search.

Let's also make sure a real freezing value works.

In [26]:
log.readings = [7, 4, -2, -8]

log.first_freezing_reading

searching for freezing reading...


-2

This pattern is important whenever a cached computation can legitimately return:

- `None`,
- `False`,
- `0`,
- an empty string,
- an empty tuple,
- or any other value that could be confused with an “empty cache” marker.

A unique sentinel object avoids that ambiguity.

### Problem 4 — Read-Only Does Not Automatically Mean Immutable

Now let's look at a subtle issue.

Suppose a class stores raw tags and exposes a read-only property containing normalized tags.

The property has no setter, so it looks read-only.

Let's build a first version.

In [27]:
class Document:
    def __init__(self, tags):
        self.tags = list(tags)

    @property
    def normalized_tags(self):
        return [
            tag.strip().lower()
            for tag in self.tags
            if tag.strip()
        ]

In [28]:
doc = Document([
    " Python ",
    "OOP",
    " Properties ",
])

doc.normalized_tags

['python', 'oop', 'properties']

This version is actually fairly safe because every property access creates a brand new list.

But that means we repeat the normalization work every time.

Let's cache the normalized tags.

In [29]:
class Document:
    def __init__(self, tags):
        self._tags = []
        self._normalized_tags = None
        self.tags = tags

    @property
    def tags(self):
        return self._tags

    @tags.setter
    def tags(self, value):
        self._tags = list(value)
        self._normalized_tags = None

    @property
    def normalized_tags(self):
        if self._normalized_tags is None:
            print("normalizing tags...")

            self._normalized_tags = [
                tag.strip().lower()
                for tag in self.tags
                if tag.strip()
            ]

        return self._normalized_tags

In [30]:
doc = Document([
    " Python ",
    "OOP",
    " Properties ",
])

doc.normalized_tags

normalizing tags...


['python', 'oop', 'properties']

In [31]:
doc.normalized_tags

['python', 'oop', 'properties']

The cache works.

But now we have created a new problem.

The property returns the exact list stored inside `_normalized_tags`.

Even though the property has no setter, a caller can mutate the returned list.

In [32]:
tags = doc.normalized_tags
tags.append("corrupted")

doc.normalized_tags

['python', 'oop', 'properties', 'corrupted']

So the property is read-only only in the sense that this is not allowed:

```python
doc.normalized_tags = [...]
```

But the object returned by the property is still mutable.

A simple solution is to cache an immutable tuple instead.

In [33]:
class Document:
    def __init__(self, tags):
        self._tags = ()
        self._normalized_tags = None
        self.tags = tags

    @property
    def tags(self):
        return self._tags

    @tags.setter
    def tags(self, value):
        value = tuple(value)

        if not all(isinstance(tag, str) for tag in value):
            raise TypeError("every tag must be a string")

        if value != self._tags:
            self._tags = value
            self._normalized_tags = None

    @property
    def normalized_tags(self):
        if self._normalized_tags is None:
            print("normalizing tags...")

            seen = set()
            result = []

            for tag in self.tags:
                normalized = tag.strip().lower()

                if normalized and normalized not in seen:
                    seen.add(normalized)
                    result.append(normalized)

            self._normalized_tags = tuple(result)

        return self._normalized_tags

In [34]:
doc = Document([
    " Python ",
    "OOP",
    "python",
    " Properties ",
])

doc.normalized_tags

normalizing tags...


('python', 'oop', 'properties')

Now callers can read the normalized tags, but they cannot append to the returned tuple.

We have also made the raw `tags` property return a tuple so callers cannot silently mutate the input data behind the class's back.

This is an important cache-design lesson.

If your cached value is mutable and you return it directly, outside code may be able to make your cache incorrect.

### Problem 5 — A Lazy Parsed Configuration

Computed properties do not have to be simple arithmetic.

They can also represent a lazily produced object.

Suppose a configuration object stores JSON text.

Parsing JSON is not usually extremely expensive, but it is a good example because parsing creates a derived Python object from the original text.

We want the following behavior:

- constructing the object stores only the text,
- parsing does not happen immediately,
- the first request for parsed data performs the parse,
- later requests reuse the parsed result,
- replacing the text invalidates the parsed result.

Let's start with the most direct version.

In [35]:
import json

class JSONConfig:
    def __init__(self, text):
        self.text = text

    @property
    def data(self):
        print("parsing JSON...")
        return json.loads(self.text)

In [36]:
config = JSONConfig('{"host": "localhost", "port": 8000}')

config.data

parsing JSON...


{'host': 'localhost', 'port': 8000}

In [37]:
config.data

parsing JSON...


{'host': 'localhost', 'port': 8000}

The JSON is parsed every time.

So now we can introduce a cache.

In [38]:
import json

class JSONConfig:
    def __init__(self, text):
        self._text = None
        self._data = None
        self.text = text

    @property
    def text(self):
        return self._text

    @text.setter
    def text(self, value):
        if not isinstance(value, str):
            raise TypeError("text must be a string")

        if value != self._text:
            self._text = value
            self._data = None

    @property
    def data(self):
        if self._data is None:
            print("parsing JSON...")
            self._data = json.loads(self.text)

        return self._data

In [39]:
config = JSONConfig('{"host": "localhost", "port": 8000}')

config.data

parsing JSON...


{'host': 'localhost', 'port': 8000}

In [40]:
config.data

{'host': 'localhost', 'port': 8000}

Now the parsing is lazy and cached.

But we have a familiar problem.

`json.loads()` normally returns mutable dictionaries and lists.

If we return the exact cached dictionary, callers can modify the cache.

In [41]:
config.data["port"] = 9999

config.data

{'host': 'localhost', 'port': 9999}

Our original JSON text still says `8000`, but the cached Python object now says `9999`.

The two representations disagree.

There are several possible policies here.

For this exercise we are going to choose a simple one:

- the internal parsed object remains private,
- the public property returns a deep copy.

This keeps the cache protected from callers.

In [42]:
import json
from copy import deepcopy

class JSONConfig:
    def __init__(self, text):
        self._text = None
        self._data = None
        self.text = text

    @property
    def text(self):
        return self._text

    @text.setter
    def text(self, value):
        if not isinstance(value, str):
            raise TypeError("text must be a string")

        if value != self._text:
            # Validate first so a bad assignment does not destroy
            # a previously valid object state.
            json.loads(value)

            self._text = value
            self._data = None

    def _parse(self):
        print("parsing JSON...")
        self._data = json.loads(self.text)

    @property
    def data(self):
        if self._data is None:
            self._parse()

        return deepcopy(self._data)

In [43]:
config = JSONConfig('{"host": "localhost", "port": 8000}')

first = config.data
first["port"] = 9999

second = config.data

first, second

parsing JSON...


({'host': 'localhost', 'port': 9999}, {'host': 'localhost', 'port': 8000})

The mutation to `first` does not affect the cached value.

This is safer, although creating a deep copy also has a cost.

That trade-off is part of API design:

- return the internal object for maximum speed,
- return an immutable representation,
- or return a copy for isolation.

### Problem 6 — Writable Computed Properties

Most of the computed properties we have written so far are read-only.

But sometimes a derived property has a natural inverse operation.

Temperature is a classic example.

We can store one canonical value internally—Celsius—and expose both Celsius and Fahrenheit.

The Fahrenheit value is computed from Celsius:

```text
F = C × 9/5 + 32
```

But the conversion also works in reverse:

```text
C = (F - 32) × 5/9
```

That means `fahrenheit` can reasonably have a setter.

In [44]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        self._celsius = float(value)

    @property
    def fahrenheit(self):
        return self.celsius * 9 / 5 + 32

In [45]:
t = Temperature(100)

t.fahrenheit

212.0

Now let's add a setter to the computed Fahrenheit property.

In [46]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        value = float(value)

        if value < -273.15:
            raise ValueError("temperature cannot be below absolute zero")

        self._celsius = value

    @property
    def fahrenheit(self):
        return self.celsius * 9 / 5 + 32

    @fahrenheit.setter
    def fahrenheit(self, value):
        value = float(value)

        celsius = (value - 32) * 5 / 9

        # Route through the canonical setter so that all validation
        # remains in one place.
        self.celsius = celsius

In [47]:
t = Temperature(100)

print(t.celsius)
print(t.fahrenheit)

100.0
212.0


In [48]:
t.fahrenheit = 32

print(t.celsius)
print(t.fahrenheit)

0.0
32.0


The useful design detail is that the Fahrenheit setter does **not** directly assign `_celsius`.

Instead it converts the value and then calls:

```python
self.celsius = celsius
```

That means the canonical Celsius setter remains responsible for validation.

This is a good example of a writable computed property.

We still store only one independent value.

The other representation is derived in both directions.

### Problem 7 — Version-Based Cache Invalidation

Manual cache invalidation works well when a class has only a few dependencies.

But imagine an object with many mutable inputs and many computed values.

It can become difficult to remember which setter should clear which cache.

One alternative is to maintain a version number.

Every real change to the object's independent state increments the version.

A cached property remembers which version produced its current value.

If the object's current version and the cache version match, the cache is valid.

If they differ, the value must be recalculated.

Let's use this idea for a 2D vector.

We will calculate its magnitude:

```text
sqrt(x² + y²)
```

In [49]:
from math import hypot

class Vector:
    def __init__(self, x, y):
        self._x = None
        self._y = None
        self._version = 0

        self._magnitude = None
        self._magnitude_version = -1

        self.x = x
        self.y = y

Now we add setters that increment `_version` only when the coordinate actually changes.

In [50]:
class Vector(Vector):
    @property
    def x(self):
        return self._x

    @x.setter
    def x(self, value):
        value = float(value)

        if value != self._x:
            self._x = value
            self._version += 1

    @property
    def y(self):
        return self._y

    @y.setter
    def y(self, value):
        value = float(value)

        if value != self._y:
            self._y = value
            self._version += 1

And now we can write the version-aware computed property.

In [51]:
class Vector(Vector):
    @property
    def magnitude(self):
        if self._magnitude_version != self._version:
            print("calculating magnitude...")

            self._magnitude = hypot(self.x, self.y)
            self._magnitude_version = self._version

        return self._magnitude

In [52]:
v = Vector(3, 4)

v.magnitude

calculating magnitude...


5.0

In [53]:
v.magnitude

5.0

The cache is valid because the version has not changed.

Now let's set `x` to the same value it already has.

In [54]:
v.x = 3

v.magnitude

5.0

There was no real state change, so the version did not change and the cache remained valid.

Now let's actually change a coordinate.

In [55]:
v.x = 6

v.magnitude

calculating magnitude...


7.211102550927978

The coordinate setter incremented the version.

The next magnitude access detected that the cache belonged to an older version and recalculated it.

Versioning is useful because the setters do not need to know exactly which caches depend on them.

They only need to say:

> the object changed

Each cached property can then decide whether its own cached version is still current.

### Problem 8 — A Lazy Text Analyzer

For the final problem we are going to combine several ideas.

We will create a `TextAnalyzer` that stores text and exposes these read-only properties:

- `words`
- `word_count`
- `unique_words`
- `most_common_word`
- `average_word_length`

We do not want to tokenize the text until one of these values is actually requested.

The first step is to store the raw text and keep the token cache empty.

In [56]:
import re

class TextAnalyzer:
    def __init__(self, text):
        self._text = None
        self._words = None
        self.text = text

    @property
    def text(self):
        return self._text

    @text.setter
    def text(self, value):
        if not isinstance(value, str):
            raise TypeError("text must be a string")

        if value != self._text:
            self._text = value
            self._words = None

Now let's add lazy tokenization.

We will keep only alphabetic words and normalize them to lowercase.

In [57]:
class TextAnalyzer(TextAnalyzer):
    def _tokenize(self):
        print("tokenizing text...")

        self._words = tuple(
            match.group(0).lower()
            for match in re.finditer(r"[A-Za-z]+", self.text)
        )

    @property
    def words(self):
        if self._words is None:
            self._tokenize()

        return self._words

In [58]:
analyzer = TextAnalyzer(
    "Properties are useful. Properties can also cache computed values."
)

analyzer.words

tokenizing text...


('properties',
 'are',
 'useful',
 'properties',
 'can',
 'also',
 'cache',
 'computed',
 'values')

In [59]:
analyzer.words

('properties',
 'are',
 'useful',
 'properties',
 'can',
 'also',
 'cache',
 'computed',
 'values')

The tokenization only happened once.

Because `words` returns a tuple, callers cannot append or remove words from our cached token sequence.

Now several other properties can build on top of the tokenized data.

In [60]:
from collections import Counter

class TextAnalyzer(TextAnalyzer):
    @property
    def word_count(self):
        return len(self.words)

    @property
    def unique_words(self):
        return frozenset(self.words)

    @property
    def most_common_word(self):
        if not self.words:
            return None

        counts = Counter(self.words)
        return counts.most_common(1)[0][0]

    @property
    def average_word_length(self):
        if not self.words:
            return None

        return sum(map(len, self.words)) / len(self.words)

In [61]:
analyzer = TextAnalyzer(
    "Properties are useful. Properties can also cache computed values."
)

print("word_count:", analyzer.word_count)
print("unique_words:", analyzer.unique_words)
print("most_common_word:", analyzer.most_common_word)
print("average_word_length:", analyzer.average_word_length)

tokenizing text...
word_count: 9
unique_words: frozenset({'properties', 'cache', 'can', 'values', 'useful', 'computed', 'are', 'also'})
most_common_word: properties
average_word_length: 6.111111111111111


This is already useful, but `most_common_word` and `average_word_length` still repeat their own calculations every time.

The tokenization is cached, but those higher-level results are not.

We can add independent caches for them.

Because they all depend on the text, changing the text must invalidate all of them.

In [62]:
_NOT_READY = object()

class TextAnalyzer:
    def __init__(self, text):
        self._text = None

        self._words = None
        self._most_common_word = _NOT_READY
        self._average_word_length = _NOT_READY

        self.text = text

    def _invalidate(self):
        self._words = None
        self._most_common_word = _NOT_READY
        self._average_word_length = _NOT_READY

    @property
    def text(self):
        return self._text

    @text.setter
    def text(self, value):
        if not isinstance(value, str):
            raise TypeError("text must be a string")

        if value != self._text:
            self._text = value
            self._invalidate()

    def _tokenize(self):
        print("tokenizing text...")

        self._words = tuple(
            match.group(0).lower()
            for match in re.finditer(r"[A-Za-z]+", self.text)
        )

    @property
    def words(self):
        if self._words is None:
            self._tokenize()

        return self._words

    @property
    def word_count(self):
        return len(self.words)

    @property
    def unique_words(self):
        return frozenset(self.words)

    @property
    def most_common_word(self):
        if self._most_common_word is _NOT_READY:
            print("calculating most common word...")

            if not self.words:
                self._most_common_word = None
            else:
                counts = Counter(self.words)
                self._most_common_word = counts.most_common(1)[0][0]

        return self._most_common_word

    @property
    def average_word_length(self):
        if self._average_word_length is _NOT_READY:
            print("calculating average word length...")

            if not self.words:
                self._average_word_length = None
            else:
                self._average_word_length = (
                    sum(map(len, self.words)) / len(self.words)
                )

        return self._average_word_length

In [63]:
analyzer = TextAnalyzer(
    "Cache invalidation is hard, but properties can help organize it."
)

print(analyzer.most_common_word)
print(analyzer.average_word_length)

calculating most common word...
tokenizing text...
cache
calculating average word length...
5.3


In [64]:
print(analyzer.most_common_word)
print(analyzer.average_word_length)

cache
5.3


No work was repeated on the second access.

Now let's replace the text.

In [65]:
analyzer.text = "One one two two two three"

print(analyzer.most_common_word)
print(analyzer.average_word_length)

calculating most common word...
tokenizing text...
two
calculating average word length...
3.3333333333333335


Changing the text invalidated all text-dependent caches.

The next accesses rebuilt only what was needed.

### Challenge Problem — Build a Cached Polynomial

Let's finish with a problem for you to solve using the same ideas.

Create a `Polynomial` class.

The constructor should receive a sequence of coefficients.

For example:

```python
Polynomial([2, -3, 5])
```

represents:

```text
2x² - 3x + 5
```

The object should also have a writable `x` property.

It should expose a read-only computed property named `value`.

Your first version should simply evaluate the polynomial every time `value` is requested.

Use Horner's method rather than explicitly calculating powers.

For the polynomial:

```text
2x² - 3x + 5
```

Horner's method rewrites the expression as:

```text
(2x - 3)x + 5
```

That avoids repeated exponentiation.

After your first version works, improve it so that:

- the computed value is cached,
- assigning the same `x` does not invalidate the cache,
- assigning a different `x` does invalidate the cache,
- replacing the coefficients invalidates the cache,
- coefficients are stored as an immutable tuple,
- the class exposes a read-only `degree` property.

Try the problem before looking at the solution below.

### Challenge Solution — Step 1

We will begin by storing the coefficients and `x`.

The coefficient setter will also reject an empty sequence because a polynomial needs at least one coefficient.

In [66]:
class Polynomial:
    def __init__(self, coefficients, x=0):
        self._coefficients = ()
        self._x = None

        self.coefficients = coefficients
        self.x = x

    @property
    def coefficients(self):
        return self._coefficients

    @coefficients.setter
    def coefficients(self, value):
        value = tuple(value)

        if not value:
            raise ValueError("coefficients cannot be empty")

        self._coefficients = value

    @property
    def x(self):
        return self._x

    @x.setter
    def x(self, value):
        self._x = value

    @property
    def degree(self):
        return len(self.coefficients) - 1

Now let's add a first, uncached implementation of `value`.

In [67]:
class Polynomial(Polynomial):
    @property
    def value(self):
        print("evaluating polynomial...")

        result = 0

        for coefficient in self.coefficients:
            result = result * self.x + coefficient

        return result

In [68]:
p = Polynomial([2, -3, 5], x=4)

p.value

evaluating polynomial...


25

In [69]:
p.value

evaluating polynomial...


25

We get the correct answer, but the polynomial is evaluated every time.

So now we add a cache.

We will use a sentinel instead of `None`.

That is slightly more general because a polynomial could theoretically evaluate to `None` if we allowed unusual coefficient types.

In [70]:
_VALUE_NOT_CACHED = object()

class Polynomial:
    def __init__(self, coefficients, x=0):
        self._coefficients = ()
        self._x = None
        self._value = _VALUE_NOT_CACHED

        self.coefficients = coefficients
        self.x = x

    def _invalidate_value(self):
        self._value = _VALUE_NOT_CACHED

    @property
    def coefficients(self):
        return self._coefficients

    @coefficients.setter
    def coefficients(self, value):
        value = tuple(value)

        if not value:
            raise ValueError("coefficients cannot be empty")

        if value != self._coefficients:
            self._coefficients = value
            self._invalidate_value()

    @property
    def x(self):
        return self._x

    @x.setter
    def x(self, value):
        if value != self._x:
            self._x = value
            self._invalidate_value()

    @property
    def degree(self):
        return len(self.coefficients) - 1

    @property
    def value(self):
        if self._value is _VALUE_NOT_CACHED:
            print("evaluating polynomial...")

            result = 0

            for coefficient in self.coefficients:
                result = result * self.x + coefficient

            self._value = result

        return self._value

In [71]:
p = Polynomial([2, -3, 5], x=4)

p.value

evaluating polynomial...


25

In [72]:
p.value

25

Now set `x` to the same value.

In [73]:
p.x = 4

p.value

25

No recalculation was needed because the setter detected that the independent state did not actually change.

Now change `x`.

In [74]:
p.x = 10

p.value

evaluating polynomial...


175

And finally, replace the coefficients.

In [75]:
p.coefficients = [1, 0, -1]

print("degree:", p.degree)
print("value:", p.value)

degree: 2
evaluating polynomial...
value: 99


The value cache was invalidated in both places where its dependencies could change:

- `x`
- `coefficients`

That is the key rule for any cached computed property:

> every mutation that can change the answer must either invalidate the cache or change the version used to validate the cache.

### Review

In this notebook we used computed properties in several different ways.

We started with simple derived values and then gradually introduced caching and invalidation.

The main patterns we used were:

1. **Direct computed properties**

   Calculate the value every time.

2. **Single cached derived object**

   Calculate several related values together and cache the combined result.

3. **Selective invalidation**

   Invalidate only the cached values affected by a particular input.

4. **Sentinel values**

   Use a unique object when `None` is itself a valid computed result.

5. **Immutable public results**

   Use tuples or other immutable objects so callers cannot corrupt cached state.

6. **Lazy computation**

   Delay expensive work until the value is actually requested.

7. **Writable derived properties**

   Allow assignment when there is a clear inverse conversion.

8. **Version-based caching**

   Track whether a cache belongs to the current object state without manually clearing it from every setter.

There is one final design question worth remembering.

Just because something *can* be implemented as a property does not mean it always *should* be.

Properties work best when attribute-style access is natural.

If obtaining the value performs something surprising—such as a network request, a database query, a long-running computation, or an external side effect—an explicit method can make the cost clearer to the caller.

### Extra Practice

Here are a few more problems you can build using the same techniques.

#### 1. `ImageMetadata`

Store width, height, and color depth.

Add read-only properties for:

- aspect ratio,
- pixel count,
- approximate uncompressed size.

Cache only what is worth caching.

#### 2. `GradeBook`

Store student scores.

Add read-only properties for:

- mean,
- median,
- highest score,
- lowest score,
- passing percentage.

Use one cached statistics summary rather than calculating every metric independently.

#### 3. `Route`

Store a sequence of 2D points.

Add read-only properties for:

- segment lengths,
- total distance,
- longest segment.

Make sure replacing the points invalidates all route metrics.

#### 4. `InventoryItem`

Store base price, markup rate, quantity, and tax rate.

Add computed properties for:

- sale price,
- inventory value before tax,
- tax amount,
- inventory value after tax.

Try implementing selective invalidation.

#### 5. `SearchResultPage`

Store raw result records.

Lazily compute:

- unique domains,
- result count by domain,
- most common domain.

Make all returned cached collections immutable.